# Azure Key Vault Secrets Import

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")

SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
TENANT_ID = os.environ["AZURE_TENANT_ID"]
os.environ["SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["TENANT_ID"] = TENANT_ID
os.environ["WORKDIR"] = "/tmp/vault"
os.environ["RESOURCE_GROUP"] = "VaultDemoRG"
os.environ["APP_NAME"] = "vaultsecretsync-mapfre"
os.environ["KEYVAULT"] = "vaultsecretsync-mapfre"


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = next((directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("Could not find the persistent .env file")
load_dotenv(ENV_FILE)

VAULT_TOKEN = os.getenv('VAULT_TOKEN')
VAULT_ADDR = os.getenv('VAULT_ADDR')

## Authenticate and confirm the Azure subscription

In [ ]:
! az login --tenant $TENANT_ID --subscription $SUBSCRIPTION_ID

In [ ]:
%%bash
az account show \
  --query '{subscription:name, subscriptionId:id, tenantId:tenantId}' \
  --output table

In [ ]:
%%bash
az provider register \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID"

In [ ]:
! az provider show \
  --namespace Microsoft.KeyVault \
  --subscription "$SUBSCRIPTION_ID" \
  --query '{namespace:namespace,state:registrationState}' \
  --output table

In [ ]:
! az group create --name $RESOURCE_GROUP --location westeurope 

In [ ]:
! az group show --name $RESOURCE_GROUP | jq -r '.id'

In [ ]:
%%bash

az keyvault create --name $KEYVAULT --resource-group $RESOURCE_GROUP --location westeurope 

## Activate Secret Import

In [ ]:
! vault write -f sys/activation-flags/secrets-import/activate

## Authorize the identity persisted by `az login`

In [ ]:
%%bash
set -euo pipefail

az account set --subscription "$SUBSCRIPTION_ID"
ACCOUNT_TYPE=$(az account show --query user.type -o tsv)
if [[ "$ACCOUNT_TYPE" == "user" ]]; then
  PRINCIPAL_ID=$(az ad signed-in-user show --query id -o tsv)
  PRINCIPAL_TYPE=User
else
  CLIENT_ID=$(az account show --query user.name -o tsv)
  PRINCIPAL_ID=$(az ad sp show --id "$CLIENT_ID" --query id -o tsv)
  PRINCIPAL_TYPE=ServicePrincipal
fi
KEYVAULT_ID=$(az keyvault show --name "$KEYVAULT" --resource-group "$RESOURCE_GROUP" --query id -o tsv)

if ! az role assignment list --assignee "$PRINCIPAL_ID" --scope "$KEYVAULT_ID" \
  --query "[?roleDefinitionName=='Key Vault Secrets Officer'] | [0].id" -o tsv | grep -q .; then
  az role assignment create \
    --assignee-object-id "$PRINCIPAL_ID" \
    --assignee-principal-type "$PRINCIPAL_TYPE" \
    --role "Key Vault Secrets Officer" \
    --scope "$KEYVAULT_ID" >/dev/null
fi

for attempt in $(seq 1 30); do
  if az keyvault secret list --vault-name "$KEYVAULT" --maxresults 1 >/dev/null 2>&1; then
    break
  fi
  [[ "$attempt" == 30 ]] && { echo "Key Vault RBAC did not propagate in time" >&2; exit 1; }
  sleep 10
done

az account show --query '{subscription:name,subscriptionId:id,tenantId:tenantId,identity:user.name}' -o table

## Create ten tagged source secrets

Values and PEM material are generated locally and are never printed by the notebook.

In [ ]:
%%bash
set -euo pipefail
umask 077

DB_PASSWORD="Db-$(openssl rand -hex 16)"
PAYMENTS_API_KEY="pay_$(openssl rand -hex 24)"
MAPS_API_KEY="maps_$(openssl rand -hex 24)"
OAUTH_SECRET="oauth_$(openssl rand -base64 32 | tr -d '\n')"
WEBHOOK_SECRET="whsec_$(openssl rand -hex 24)"
JWT_PRIVATE_KEY=$(openssl genpkey -algorithm EC -pkeyopt ec_paramgen_curve:P-256 2>/dev/null)
SSH_PRIVATE_KEY=$(openssl genpkey -algorithm RSA -pkeyopt rsa_keygen_bits:2048 2>/dev/null)
TLS_CERTIFICATE=$(openssl req -x509 -newkey rsa:2048 -nodes -days 30 \
  -subj '/CN=secrets-import-demo.local' -keyout /dev/null -out /dev/stdout 2>/dev/null)

set_secret() {
  local name=$1 value=$2 category=$3 format=$4
  az keyvault secret set --vault-name "$KEYVAULT" --name "$name" --value "$value" \
    --tags importable=true migration=azure-secrets-import owner=mapfre \
      environment=demo category="$category" format="$format" --output none
}

set_secret demo-db-username 'app_import_user' database username
set_secret demo-db-password "$DB_PASSWORD" database password
set_secret demo-payments-api-key "$PAYMENTS_API_KEY" api api-key
set_secret demo-maps-api-key "$MAPS_API_KEY" api api-key
set_secret demo-oauth-client-secret "$OAUTH_SECRET" oauth client-secret
set_secret demo-webhook-signing-secret "$WEBHOOK_SECRET" webhook signing-secret
set_secret demo-jwt-private-key "$JWT_PRIVATE_KEY" crypto pem
set_secret demo-ssh-private-key "$SSH_PRIVATE_KEY" ssh pem
set_secret demo-tls-certificate "$TLS_CERTIFICATE" tls certificate-pem
set_secret demo-connection-string "Server=db.demo.internal;User=app_import_user;Password=${DB_PASSWORD}" database connection-string

unset DB_PASSWORD PAYMENTS_API_KEY MAPS_API_KEY OAUTH_SECRET WEBHOOK_SECRET
unset JWT_PRIVATE_KEY SSH_PRIVATE_KEY TLS_CERTIFICATE

az keyvault secret list --vault-name "$KEYVAULT" \
  --query "[?tags.migration=='azure-secrets-import'].{name:name,category:tags.category,format:tags.format,importable:tags.importable}" \
  --output table

## Plan and apply Secrets Import

Only secrets tagged with `importable=true` and `migration=azure-secrets-import` are selected.

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$WORKDIR"
source "$WORKDIR/azure-import-sp.env"
cat > "$WORKDIR/azure-import.hcl" <<EOF
source_azure {
  name          = "azure-key-vault"
  key_vault_uri = "https://${KEYVAULT}.vault.azure.net/"
  tenant_id     = "${TENANT_ID}"
  client_id     = "${APP_ID}"
  credentials_file = "$WORKDIR/azure-import-client-secret"
}

destination_vault {
  name    = "vault-kv"
  address = "${VAULT_ADDR}"
  mount   = "azure-import"
}

mapping {
  name        = "tagged-azure-secrets"
  source      = "azure-key-vault"
  destination = "vault-kv"
  filter      = "Secret.Tags.importable == \"true\" and Secret.Tags.migration == \"azure-secrets-import\""
}
EOF

echo "Import plan written to $WORKDIR/azure-import.hcl"

In [ ]:
%%bash
set -euo pipefail

: "${VAULT_ADDR:?VAULT_ADDR is required}"
: "${VAULT_TOKEN:?VAULT_TOKEN is required}"
vault version | grep -q -- '+ent' || {
  echo 'Vault Enterprise CLI is required for operator import.' >&2
  exit 1
}

for attempt in {1..18}; do
  if vault operator import -config="$WORKDIR/azure-import.hcl" plan; then
    break
  fi
  if [[ "$attempt" -eq 18 ]]; then
    echo 'Azure RBAC did not become effective within three minutes.' >&2
    exit 1
  fi
  echo "Waiting for Azure RBAC propagation ($attempt/18)..."
  sleep 10
done
vault operator import -config="$WORKDIR/azure-import.hcl" -auto-create -auto-approve apply

## Verify names without exposing secret values

In [ ]:
%%bash
set -euo pipefail

EXPECTED_NAMES=$(az keyvault secret list --vault-name "$KEYVAULT" \
  --query "[?tags.migration=='azure-secrets-import'].name" -o tsv | sort)
IMPORTED_NAMES=$(vault kv list -format=json azure-import | jq -r '.[]' | sort)
diff <(printf '%s\n' "$EXPECTED_NAMES") <(printf '%s\n' "$IMPORTED_NAMES")
COUNT=$(printf '%s\n' "$IMPORTED_NAMES" | grep -c .)
[[ "$COUNT" == 10 ]]
printf 'Imported %s secrets into azure-import/:\n%s\n' "$COUNT" "$IMPORTED_NAMES"

## Verify Azure tags against Vault custom metadata

The utility below verifies that every Azure tag exists with the same value in Vault KV v2 `custom_metadata`. Vault-specific metadata such as `import-source` and `operation` is allowed.

In [ ]:
%%bash
set -euo pipefail

verify_secret_tags_match_vault_metadata() {
  local key_vault_name=$1
  local vault_mount=$2
  local secret_name=$3
  local azure_tags vault_metadata mismatches

  azure_tags=$(az keyvault secret show \
    --vault-name "$key_vault_name" \
    --name "$secret_name" \
    --query tags -o json | jq '. // {}')
  vault_metadata=$(vault kv metadata get -format=json \
    "$vault_mount/$secret_name" | jq '.data.custom_metadata // {}')

  mismatches=$(jq -n \
    --argjson azure "$azure_tags" \
    --argjson vault "$vault_metadata" \
    '$azure | to_entries | map(select($vault[.key] != .value))')

  if [[ $(jq 'length' <<<"$mismatches") -ne 0 ]]; then
    echo "ERROR: tag mismatch for $secret_name" >&2
    jq -r '.[] | "  \(.key): Azure=\(.value), Vault=missing-or-different"' \
      <<<"$mismatches" >&2
    return 1
  fi
  printf 'OK %s\n' "$secret_name"
}

VERIFIED_TAGS=0
while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  verify_secret_tags_match_vault_metadata "$KEYVAULT" azure-import "$secret_name"
  VERIFIED_TAGS=$((VERIFIED_TAGS + 1))
done < <(az keyvault secret list --vault-name "$KEYVAULT" \
  --query "[?tags.migration=='azure-secrets-import' && tags.importable=='true'].name" \
  -o tsv | sort)

[[ "$VERIFIED_TAGS" -eq 10 ]]
echo "Verified Azure tags against Vault custom metadata for $VERIFIED_TAGS secrets."

## Sync the imported secrets back to Azure

This section reuses the temporary Azure SPN created above and the Secrets Sync pattern from `5_Secret_Sync_Azure_CLI_SPN.ipynb`. Because every imported KV secret contains one key named `value`, `secret-key` granularity restores the original raw Azure value. The name template consumes `.SecretKey` as required by Vault but emits exactly `.SecretBaseName`, preserving every original secret name.

In [ ]:
%%bash
set -euo pipefail

SYNC_DESTINATION=azure-import-roundtrip
source "$WORKDIR/azure-import-sp.env"
CLIENT_SECRET=$(<"$WORKDIR/azure-import-client-secret")
VAULT_URI=$(az keyvault show \
  --name "$KEYVAULT" \
  --resource-group "$RESOURCE_GROUP" \
  --query properties.vaultUri -o tsv)

vault write -f sys/activation-flags/secrets-sync/activate >/dev/null || true
vault write "sys/sync/destinations/azure-kv/$SYNC_DESTINATION" \
  key_vault_uri="$VAULT_URI" \
  client_id="$APP_ID" \
  client_secret="$CLIENT_SECRET" \
  tenant_id="$TENANT_ID" \
  granularity=secret-key \
  secret_name_template='{{ $unused := .SecretKey }}{{ .SecretBaseName }}'

unset CLIENT_SECRET

In [ ]:
%%bash
set -euo pipefail

SYNC_DESTINATION=azure-import-roundtrip
while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  DATA_KEYS=$(vault kv get -format=json "azure-import/$secret_name" | \
    jq -r '.data.data | keys | join(",")')
  if [[ "$DATA_KEYS" != "value" ]]; then
    echo "ERROR: $secret_name does not contain exactly the imported key 'value'." >&2
    exit 1
  fi
  vault write \
    "sys/sync/destinations/azure-kv/$SYNC_DESTINATION/associations/set" \
    mount=azure-import \
    secret_name="$secret_name" >/dev/null
  echo "Associated $secret_name"
done < <(vault kv list -format=json azure-import | jq -r '.[]' | sort)

## Verify round-trip names, values, and sync status

The check compares secret-name sets and SHA-256 digests without printing secret values.

In [ ]:
%%bash
set -euo pipefail

SYNC_DESTINATION=azure-import-roundtrip
VAULT_NAMES=$(vault kv list -format=json azure-import | jq -r '.[]' | sort)
AZURE_NAMES=$(az keyvault secret list --vault-name "$KEYVAULT" \
  --query '[].name' -o tsv | sort)
diff <(printf '%s\n' "$VAULT_NAMES") <(printf '%s\n' "$AZURE_NAMES")

VERIFIED_VALUES=0
while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  VAULT_VALUE=$(vault kv get -field=value "azure-import/$secret_name")
  AZURE_VALUE=$(az keyvault secret show \
    --vault-name "$KEYVAULT" --name "$secret_name" --query value -o tsv)
  VAULT_DIGEST=$(printf '%s' "$VAULT_VALUE" | \
    openssl dgst -sha256 -r | awk '{print $1}')
  AZURE_DIGEST=$(printf '%s' "$AZURE_VALUE" | \
    openssl dgst -sha256 -r | awk '{print $1}')
  unset VAULT_VALUE AZURE_VALUE
  [[ "$VAULT_DIGEST" == "$AZURE_DIGEST" ]] || {
    echo "ERROR: value mismatch for $secret_name" >&2
    exit 1
  }
  VERIFIED_VALUES=$((VERIFIED_VALUES + 1))
  echo "OK $secret_name"
done <<<"$VAULT_NAMES"

ASSOCIATIONS=$(vault read -format=json \
  "sys/sync/destinations/azure-kv/$SYNC_DESTINATION/associations")
SYNCED_COUNT=$(jq '[.data.associated_secrets[] | select(.sync_status == "SYNCED")] | length' \
  <<<"$ASSOCIATIONS")
TOTAL_COUNT=$(jq '.data.associated_secrets | length' <<<"$ASSOCIATIONS")
[[ "$VERIFIED_VALUES" -eq 10 && "$SYNCED_COUNT" -eq 10 && "$TOTAL_COUNT" -eq 10 ]]
echo "Verified 10 identical names, values, and SYNCED associations in Azure Key Vault."

## Optional cleanup

Set `RUN_AZURE_IMPORT_CLEANUP=true` to delete the ten Azure source secrets and disable `azure-import/`.

In [ ]:
%%bash
set -euo pipefail
RUN_AZURE_IMPORT_CLEANUP=true
#RUN_AZURE_IMPORT_CLEANUP=${RUN_AZURE_IMPORT_CLEANUP:-false}
if [[ "$RUN_AZURE_IMPORT_CLEANUP" != "true" ]]; then
  echo "Cleanup skipped. Set RUN_AZURE_IMPORT_CLEANUP=true to execute it."
  exit 0
fi

SYNC_DESTINATION_PATH="sys/sync/destinations/azure-kv/azure-import-roundtrip"
if vault read "$SYNC_DESTINATION_PATH" >/dev/null 2>&1; then
  vault delete "$SYNC_DESTINATION_PATH" purge=true
  for attempt in {1..24}; do
    if ! vault read "$SYNC_DESTINATION_PATH" >/dev/null 2>&1; then
      break
    fi
    if [[ "$attempt" -eq 24 ]]; then
      echo 'ERROR: Secrets Sync destination was not purged.' >&2
      exit 1
    fi
    sleep 5
  done
fi

while IFS= read -r secret_name; do
  [[ -z "$secret_name" ]] && continue
  az keyvault secret delete --vault-name "$KEYVAULT" --name "$secret_name" --output none
done < <(az keyvault secret list --vault-name "$KEYVAULT" \
  --query "[?tags.migration=='azure-secrets-import'].name" -o tsv)

if vault secrets list -format=json | jq -e 'has("azure-import/")' >/dev/null; then
  vault secrets disable azure-import/
fi
if [[ -f "$WORKDIR/azure-import-sp.env" ]]; then
  source "$WORKDIR/azure-import-sp.env"
  az ad app delete --id "$APP_ID"
fi
if [[ -f "$WORKDIR/azure-import-client-secret" ]]; then
  : > "$WORKDIR/azure-import-client-secret"
fi
echo "Azure source secrets deleted and azure-import/ disabled."